# Part A — Montréal Tow-Risk: Data Preparation

**Your role:** Build a clean dataset for the borough-level towing regression. This notebook follows Step 2 of your team's final analysis plan.

**Inputs:** 7 annual *MONTREAL INTL A* daily weather CSVs (2020–2026) and 1 file of Montréal towing records (`remorquages.csv`).

**Output:** `data/processed/tow_risk_clean.csv`, containing one row per day and borough for Jan 1, 2021–Jun 30, 2026.

Run the cells **top to bottom**, and inspect their printed checks. If a cell reports an unexpected column name or borough category, inspect the raw file before proceeding.

## Step 1 — Folder structure

Put this notebook in the root of `Tow_Risk_Project`, alongside `data/raw/` with the eight CSV files. Rename the weather files `montreal_weather_2020.csv` through `montreal_weather_2026.csv` and the towing file `remorquages.csv`. Do not edit the originals.

Install requirements in the VS Code terminal:

```bash
python3 -m pip install pandas numpy matplotlib ipykernel
```

## Step 2 — Import libraries and check that all eight CSVs are present

In [3]:
from pathlib import Path
import re
import unicodedata
import pandas as pd
import numpy as np

RAW = Path.cwd()
OUTPUT = Path.cwd() / "data" / "processed"
OUTPUT.mkdir(parents=True, exist_ok=True)

weather_paths = [RAW / f"montreal_weather_{y}.csv" for y in range(2020, 2027)]
tow_path = RAW / "remorquages.csv"
missing_files = [str(path) for path in [tow_path, *weather_paths] if not path.exists()]
assert not missing_files, f"Missing files: {missing_files}. Check names and folder locations."
print("All 8 input CSVs were found.")

All 8 input CSVs were found.


## Step 3 — Load towing records (keep original fields as text)

In [4]:
def read_csv_flexible(path, *, all_text=False):
    """Read a CSV, handling common UTF-8 or Windows French text encodings."""
    last_error = None
    for encoding in ("utf-8-sig", "cp1252"):
        try:
            return pd.read_csv(
                path, encoding=encoding,
                dtype=str if all_text else None,
                low_memory=False
            )
        except UnicodeDecodeError as error:
            last_error = error
    raise last_error

tow_raw = read_csv_flexible(tow_path, all_text=True)
tow_raw.columns = tow_raw.columns.str.strip()
print("Original towing rows:", len(tow_raw))
print("Original columns:", tow_raw.columns.tolist())
display(tow_raw.head(3))

required_tow = {"DATE_ORIGINE", "ARRONDISSEMENT_ORIGINE"}
assert required_tow.issubset(tow_raw.columns), (
    "Cannot find the expected towing columns. Inspect the printed column names "
    "(and whether the file is separated by semicolons)."
)

Original towing rows: 529429
Original columns: ['DATE_ORIGINE', 'LONGITUDE_ORIGINE', 'LATITUDE_ORIGINE', 'RUE_ORIGINE', 'SECTEUR_ORIGINE', 'ARRONDISSEMENT_ORIGINE', 'DATE_DESTINATION', 'LONGITUDE_DESTINATION', 'LATITUDE_DESTINATION', 'RUE_DESTINATION', 'SECTEUR_DESTINATION', 'ARRONDISSEMENT_DESTINATION', 'MOTIF_REMORQUAGE']


,DATE_ORIGINE,LONGITUDE_ORIGINE,LATITUDE_ORIGINE,RUE_ORIGINE,SECTEUR_ORIGINE,ARRONDISSEMENT_ORIGINE,DATE_DESTINATION,LONGITUDE_DESTINATION,LATITUDE_DESTINATION,RUE_DESTINATION,SECTEUR_DESTINATION,ARRONDISSEMENT_DESTINATION,MOTIF_REMORQUAGE
0,2015-11-16T13:13:00,-73.5971179862804,45.5566266494817,9e avenue,NaN,Villeray-Saint-Michel - Parc-Extension,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-11-16T19:35:00,-73.5693411139059,45.4931103649656,rue de Versailles,NaN,Ville-Marie,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-11-16T20:09:00,-73.5692271519482,45.4930464058777,rue de Versailles,NaN,Ville-Marie,2015-11-16T20:10:00,-73.5682681863395,45.4930769112417,rue Saint-Jacques,NaN,Ville-Marie,Constat d'infraction


## Step 4 — Clean dates and borough names

The official towing data defines **`DATE_ORIGINE`** as the towing event date and **`ARRONDISSEMENT_ORIGINE`** as the borough where the vehicle was originally parked. We use **origin**, not destination.

Keep only 2021-01-01 through 2026-06-30 and the 17 boroughs chosen in the analysis plan. Exclude the two boroughs with very few tows and exclude separate municipalities or citywide labels.

**Important:** Inspect the *unmatched raw borough labels* in the printed output below; if a genuine Montréal borough appears there because its spelling differs, fix the mapping before moving on.

In [5]:
tow = tow_raw[["DATE_ORIGINE", "ARRONDISSEMENT_ORIGINE"]].copy()
tow["date"] = pd.to_datetime(
    tow["DATE_ORIGINE"].str.strip().str[:10],
    format="%Y-%m-%d", errors="coerce"
)
print("Rows with unparseable towing dates:", tow["date"].isna().sum())
tow = tow.loc[tow["date"].between("2021-01-01", "2026-06-30")].copy()

boroughs = [
    "Ahuntsic-Cartierville", "Anjou", "Côte-des-Neiges–Notre-Dame-de-Grâce",
    "Lachine", "LaSalle", "Le Plateau-Mont-Royal", "Le Sud-Ouest",
    "Mercier–Hochelaga-Maisonneuve", "Montréal-Nord", "Outremont",
    "Rivière-des-Prairies–Pointe-aux-Trembles", "Rosemont–La Petite-Patrie",
    "Saint-Laurent", "Saint-Léonard", "Verdun", "Ville-Marie",
    "Villeray–Saint-Michel–Parc-Extension"
]

def name_key(name):
    if pd.isna(name):
        return ""
    name = unicodedata.normalize("NFKD", str(name).strip())
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = name.replace("’", "'").replace("–", "-").replace("—", "-")
    name = re.sub(r"\s*-\s*", "-", name)
    name = re.sub(r"\s+", " ", name)
    return name.lower().strip()

canonical = {name_key(name): name for name in boroughs}
# Common alternative spelling for the borough whose official name begins "Le".
canonical[name_key("Plateau-Mont-Royal")] = "Le Plateau-Mont-Royal"

tow["borough"] = tow["ARRONDISSEMENT_ORIGINE"].map(
    lambda value: canonical.get(name_key(value))
)

print("Most common unmatched raw borough labels (review these):")
print(tow.loc[tow["borough"].isna(), "ARRONDISSEMENT_ORIGINE"].value_counts().head(30).to_string())
print("\nTows retained by borough:")
print(tow["borough"].value_counts(dropna=True).sort_index().to_string())
print("\nNumber of boroughs with at least one tow:", tow["borough"].nunique())
assert tow["borough"].nunique() == 17, (
    "Not all 17 boroughs are represented. Check the unmatched labels above."
)
tow = tow.dropna(subset=["borough"])[["date", "borough"]].copy()

Rows with unparseable towing dates: 0
Most common unmatched raw borough labels (review these):
ARRONDISSEMENT_ORIGINE
Pierrefonds - Roxboro              110
Westmount                           92
L'Île-Bizard - Sainte-Geneviève     64
Côte-Saint-Luc                      59
Mont-Royal                          45
Montréal-Est                        17
Hampstead                           13
Montréal-Ouest                       8
Dorval                               2
Kirkland                             2
Ville de Montréal                    1
Dollard-des-Ormeaux                  1

Tows retained by borough:
borough
Ahuntsic-Cartierville                       19861
Anjou                                        4145
Côte-des-Neiges–Notre-Dame-de-Grâce         27394
LaSalle                                     10432
Lachine                                      4384
Le Plateau-Mont-Royal                       35664
Le Sud-Ouest                                20847
Mercier–Hochelaga-Maisonneuve

## Step 5 — Count tows and add zero-tow days

In [6]:
first_day = pd.Timestamp("2021-01-01")
last_day = pd.Timestamp("2026-06-30")
all_dates = pd.date_range(first_day, last_day, freq="D")
all_rows = pd.MultiIndex.from_product(
    [all_dates, boroughs], names=["date", "borough"]
)

daily_tows = (
    tow.groupby(["date", "borough"]).size()
    .reindex(all_rows, fill_value=0)
    .rename("tows")
    .reset_index()
)

print("Rows after creating the full daily grid:", len(daily_tows))
print("Zero-tow days (borough-day rows):", (daily_tows["tows"] == 0).sum())
display(daily_tows.head())
assert len(daily_tows) == 17 * len(all_dates) == 34119

Rows after creating the full daily grid: 34119
Zero-tow days (borough-day rows): 14379


,date,borough,tows
0,2021-01-01,Ahuntsic-Cartierville,0
1,2021-01-01,Anjou,0
2,2021-01-01,Côte-des-Neiges–Notre-Dame-de-Grâce,0
3,2021-01-01,Lachine,0
4,2021-01-01,LaSalle,0


## Step 6 — Load the seven annual weather CSVs

The expected Environment Canada columns are `Date/Time`, `Total Snow (cm)`, `Total Rain (mm)`, and `Mean Temp (°C)`. We keep only those fields and validate year-by-year counts.

In [7]:
weather_pieces = []
required_weather = {"Date/Time", "Total Snow (cm)", "Total Rain (mm)", "Mean Temp (°C)"}

for year, path in zip(range(2020, 2027), weather_paths):
    part = read_csv_flexible(path)
    part.columns = part.columns.str.strip()
    absent = required_weather - set(part.columns)
    assert not absent, f"{path.name}: missing columns {absent}; available: {part.columns.tolist()}"

    part = part[list(required_weather)].copy()
    part = part.rename(columns={
        "Date/Time": "date",
        "Total Snow (cm)": "snow_cm",
        "Total Rain (mm)": "rain_mm",
        "Mean Temp (°C)": "mean_temp_c"
    })
    part["date"] = pd.to_datetime(part["date"], errors="coerce").dt.normalize()
    part = part.loc[part["date"].dt.year == year].copy()
    weather_pieces.append(part)
    print(f"{year}: {len(part)} daily weather rows")

weather = pd.concat(weather_pieces, ignore_index=True)
weather = weather.dropna(subset=["date"]).sort_values("date")
print("Duplicate weather dates:", weather["date"].duplicated().sum())
weather = weather.drop_duplicates(subset=["date"], keep="first")

for col in ["snow_cm", "rain_mm", "mean_temp_c"]:
    weather[col] = pd.to_numeric(weather[col], errors="coerce")

display(weather.head())

2020: 366 daily weather rows
2021: 365 daily weather rows
2022: 365 daily weather rows
2023: 365 daily weather rows
2024: 366 daily weather rows
2025: 365 daily weather rows
2026: 365 daily weather rows
Duplicate weather dates: 0


,snow_cm,date,rain_mm,mean_temp_c
0,0.8,2020-01-01,0.0,0.2
1,0.0,2020-01-02,0.0,0.8
2,0.0,2020-01-03,0.0,2.9
3,0.4,2020-01-04,0.0,-0.2
4,0.0,2020-01-05,0.0,-7.0


## Step 7 — Fill missing weather and construct the daily timeline

Follow the team plan: blank snow and rain → **0**; blank mean temperature → **interpolate** from neighboring days. Weather starts in 2020 so that early 2021 lagged values can be computed. Check the missing-data counts carefully; if an entire month/year is missing, don't silently interpret that as zero snowfall.

In [8]:
weather = weather.set_index("date").reindex(
    pd.date_range("2020-01-01", "2026-06-30", freq="D", name="date")
)
print("Missing values BEFORE filling:")
print(weather.isna().sum())

weather["snow_cm"] = weather["snow_cm"].fillna(0)
weather["rain_mm"] = weather["rain_mm"].fillna(0)
weather["mean_temp_c"] = weather["mean_temp_c"].interpolate(
    method="time", limit_direction="both"
)

assert weather[["snow_cm", "rain_mm", "mean_temp_c"]].notna().all().all(), (
    "Missing weather remains. Inspect the source CSV files."
)
print("Weather prepared. Date span:", weather.index.min().date(), "to", weather.index.max().date())

Missing values BEFORE filling:
snow_cm         8
rain_mm         9
mean_temp_c    42
dtype: int64
Weather prepared. Date span: 2020-01-01 to 2026-06-30


## Step 8 — Create **past-only** weather features

For a towing date **Jan 10**, past seven days means **Jan 3–9**. Every feature has a lag (`shift`) before rolling, to avoid using today's or future weather.

- `snow_7c` = previous seven days' snowfall, capped at **30 cm**
- `snow_8_14` = snowfall from **8–14 days ago**
- `snow_1` = yesterday's snowfall
- `rain_1` = yesterday's rainfall
- `temp_7` = mean temperature across previous seven days

In [9]:
weather["snow_7c"] = (
    weather["snow_cm"].shift(1).rolling(window=7, min_periods=7).sum()
    .clip(upper=30)
)
weather["snow_8_14"] = weather["snow_cm"].shift(8).rolling(7, min_periods=7).sum()
weather["snow_1"] = weather["snow_cm"].shift(1)
weather["rain_1"] = weather["rain_mm"].shift(1)
weather["temp_7"] = weather["mean_temp_c"].shift(1).rolling(7, min_periods=7).mean()

weather_features = weather.reset_index()[
    ["date", "snow_7c", "snow_8_14", "snow_1", "rain_1", "temp_7"]
]
display(weather_features.loc[
    weather_features["date"].between("2025-01-01", "2025-01-07")
])

,date,snow_7c,snow_8_14,snow_1,rain_1,temp_7
1827,2025-01-01,0.0,15.2,0.0,0.0,-4.000000
1828,2025-01-02,5.6,13.6,5.6,1.2,-2.200000
1829,2025-01-03,5.6,13.4,0.0,0.0,-0.642857
1830,2025-01-04,6.0,12.6,0.4,0.0,-0.042857
1831,2025-01-05,6.8,12.6,0.8,0.0,-0.785714
1832,2025-01-06,6.8,12.6,0.0,0.2,-2.785714
1833,2025-01-07,7.2,5.6,0.4,0.0,-5.571429


## Step 9 — Merge towing with weather and add calendar features

The same daily Montréal weather is used for all 17 boroughs, because the model has only one weather station.

In [10]:
data = daily_tows.merge(
    weather_features, on="date", how="left", validate="many_to_one"
)
data["dow"] = data["date"].dt.dayofweek   # Monday=0, Sunday=6
data["month"] = data["date"].dt.month
data["trend"] = (data["date"] - pd.Timestamp("2021-01-01")).dt.days / 365.25
data["moving_day"] = (
    (data["date"].dt.month == 7) & (data["date"].dt.day == 1)
).astype(int)

columns = [
    "date", "borough", "tows", "snow_7c", "snow_8_14", "snow_1",
    "rain_1", "temp_7", "dow", "month", "trend", "moving_day"
]
data = data[columns].sort_values(["date", "borough"]).reset_index(drop=True)
display(data.head())

,date,borough,tows,snow_7c,snow_8_14,snow_1,rain_1,temp_7,dow,month,trend,moving_day
0,2021-01-01,Ahuntsic-Cartierville,0,14.0,10.8,0.0,0.8,-1.828571,4,1,0.0,0
1,2021-01-01,Anjou,0,14.0,10.8,0.0,0.8,-1.828571,4,1,0.0,0
2,2021-01-01,Côte-des-Neiges–Notre-Dame-de-Grâce,0,14.0,10.8,0.0,0.8,-1.828571,4,1,0.0,0
3,2021-01-01,LaSalle,0,14.0,10.8,0.0,0.8,-1.828571,4,1,0.0,0
4,2021-01-01,Lachine,0,14.0,10.8,0.0,0.8,-1.828571,4,1,0.0,0


## Step 10 — Quality checks (do these before sharing)

Expected: **34,119 rows**, **12 columns**, 17 boroughs, complete daily dates, no nulls, unique borough/day combinations. Full-year citywide tows should be in the general range suggested in the analysis plan (roughly 34k–64k). Note that **2026 is only a half-year**.

In [11]:
print("Shape:", data.shape)
print("Date range:", data["date"].min().date(), "to", data["date"].max().date())
print("Boroughs:", data["borough"].nunique())
print("Duplicated borough/date rows:", data.duplicated(["date", "borough"]).sum())
print("Missing fields by column:")
print(data.isna().sum())

print("\nYearly towing totals (2026 is partial):")
print(data.groupby(data["date"].dt.year)["tows"].sum())

assert data.shape == (34119, 12)
assert data["borough"].nunique() == 17
assert not data.duplicated(["date", "borough"]).any()
assert not data.isna().any().any()
assert data["snow_7c"].between(0,30).all()
assert data["tows"].ge(0).all()

# Review an actual storm to confirm the lagged features rise AFTER snowfall.
storm_series = weather.loc["2021-01-01":"2026-06-30", "snow_cm"]
storm_date = storm_series.idxmax()
print("\nLargest one-day snowfall in analysis period:", storm_date.date())
display(weather.loc[
    storm_date - pd.Timedelta(days=2):storm_date + pd.Timedelta(days=8),
    ["snow_cm", "snow_1", "snow_7c", "snow_8_14"]
])
print("Quality checks passed. Visually inspect the storm table and yearly totals above.")

Shape: (34119, 12)
Date range: 2021-01-01 to 2026-06-30
Boroughs: 17
Duplicated borough/date rows: 0
Missing fields by column:
date          0
borough       0
tows          0
snow_7c       0
snow_8_14     0
snow_1        0
rain_1        0
temp_7        0
dow           0
month         0
trend         0
moving_day    0
dtype: int64

Yearly towing totals (2026 is partial):
date
2021    43248
2022    49765
2023    45377
2024    34210
2025    63965
2026    45676
Name: tows, dtype: int64

Largest one-day snowfall in analysis period: 2025-02-16


/var/folders/s2/ygqlz82918128yd7slx7py9c0000gn/T/ipykernel_66389/1761026279.py:23: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  storm_date - pd.Timedelta(days=2):storm_date + pd.Timedelta(days=8),


,snow_cm,snow_1,snow_7c,snow_8_14
date,,,,
2025-02-14,0.0,34.0,30.0,16.2
2025-02-15,1.6,0.0,30.0,17.8
2025-02-16,36.8,1.6,30.0,17.8
2025-02-17,2.2,36.8,30.0,20.0
2025-02-18,0.0,2.2,30.0,13.4
2025-02-19,0.0,0.0,30.0,13.6
2025-02-20,0.8,0.0,30.0,13.6
2025-02-21,0.0,0.8,30.0,38.6
2025-02-22,0.2,0.0,30.0,37.0


Quality checks passed. Visually inspect the storm table and yearly totals above.


In [12]:
# Compare towing counts by month and year
monthly_tows = (
    data.groupby([
        data["date"].dt.year.rename("year"),
        data["date"].dt.month.rename("month")
    ])["tows"]
    .sum()
    .unstack("year")
)

display(monthly_tows)

# Compare January–June across years
jan_jun = data[data["date"].dt.month <= 6]

print("January–June towing totals:")
print(jan_jun.groupby(jan_jun["date"].dt.year)["tows"].sum())

year,2021,2022,2023,2024,2025,2026
month,,,,,,
1,12937.0,6819.0,12638.0,11157.0,10519.0,27389.0
2,11036.0,17372.0,4745.0,1074.0,19310.0,7988.0
3,4688.0,4681.0,9347.0,1298.0,1570.0,6011.0
4,304.0,705.0,687.0,1241.0,1039.0,1092.0
5,399.0,961.0,1114.0,1508.0,1425.0,1778.0
6,486.0,1470.0,1720.0,1760.0,1597.0,1418.0
7,613.0,912.0,957.0,1354.0,1435.0,NaN
8,1189.0,984.0,1307.0,1502.0,1369.0,NaN
9,1002.0,1298.0,1673.0,1792.0,1786.0,NaN


January–June towing totals:
date
2021    29850
2022    32008
2023    30251
2024    18038
2025    35460
2026    45676
Name: tows, dtype: int64


In [13]:
# Investigate January 2026 towing activity
jan_2026 = data[
    (data["date"] >= "2026-01-01") &
    (data["date"] <= "2026-01-31")
]

# Top 10 days with the most tows
daily_jan = (
    jan_2026.groupby("date")["tows"]
    .sum()
    .sort_values(ascending=False)
)

print("Top 10 towing days in January 2026:")
print(daily_jan.head(10))

# Towing counts by borough
print("\nJanuary 2026 towing by borough:")
print(
    jan_2026.groupby("borough")["tows"]
    .sum()
    .sort_values(ascending=False)
)

Top 10 towing days in January 2026:
date
2026-01-18    3418
2026-01-08    2872
2026-01-19    2659
2026-01-09    2502
2026-01-28    2466
2026-01-29    2422
2026-01-17    1870
2026-01-30    1347
2026-01-20    1273
2026-01-02    1110
Name: tows, dtype: int64

January 2026 towing by borough:
borough
Le Plateau-Mont-Royal                       3210
Ville-Marie                                 2819
Rosemont–La Petite-Patrie                   2715
Mercier–Hochelaga-Maisonneuve               2707
Côte-des-Neiges–Notre-Dame-de-Grâce         2473
Villeray–Saint-Michel–Parc-Extension        2213
Ahuntsic-Cartierville                       2137
Le Sud-Ouest                                2115
LaSalle                                     1151
Verdun                                      1072
Montréal-Nord                               1024
Saint-Laurent                                968
Rivière-des-Prairies–Pointe-aux-Trembles     824
Saint-Léonard                                780
Lachine          

## Step 11 — Export the single CSV for B, C, D

Provide your teammates with the file **`data/processed/tow_risk_clean.csv`**. Keep the `.ipynb` as your reproducible data-cleaning code.

In [14]:
output_path = OUTPUT / "tow_risk_clean.csv"
data.to_csv(output_path, index=False, encoding="utf-8-sig")
print("SAVED:", output_path.resolve())
print("Rows:", len(data), "Columns:", len(data.columns))

SAVED: /Users/minli/Documents/Tow_Risk_Project/data/processed/tow_risk_clean.csv
Rows: 34119 Columns: 12


## Part A deliverables

1. The shared file `tow_risk_clean.csv` with 12 columns.
2. This notebook, including displayed verification outputs.
3. A ~0.75-page report section covering raw data sources, time range, borough exclusions, date parsing, zero-tow days, weather missingness, and lagged features.
4. One one-minute slide about the preparation pipeline and its limitations (e.g., one weather station for all boroughs).

**Reminder:** You're not responsible for the model itself. B handles exploration, C handles regression, D handles validation.